# HH Sport Cub S2 Dynamics (CasADi)

Fast dynamics implementation for HH Sport Cub S2 using CasADi for automatic differentiation and code generation.

**Aircraft**: HH Sport Cub S2 (4-channel control)
**Control**: [Throttle, Aileron, Elevator, Rudder]
**Mass**: 0.057 kg
**Wingspan**: 0.617 m

In [ ]:
import casadi as ca
import numpy as np

# Define smooth approximation functions in CasADi
def smooth_max_casadi(a, b, epsilon=1e-3):
    """Smooth approximation of max(a,b): (a + b + sqrt((a-b)² + ε²))/2"""
    return (a + b + ca.sqrt((a - b)**2 + epsilon**2)) / 2

def smooth_min_casadi(a, b, epsilon=1e-3):
    """Smooth approximation of min(a,b): (a + b - sqrt((a-b)² + ε²))/2"""
    return (a + b - ca.sqrt((a - b)**2 + epsilon**2)) / 2

def smooth_clamp_casadi(x, min_val, max_val, epsilon=1e-3):
    """Smooth approximation of clamping x to [min_val, max_val]"""
    return smooth_min_casadi(smooth_max_casadi(x, min_val, epsilon), max_val, epsilon)

def smooth_abs_casadi(x, epsilon=1e-6):
    """Smooth approximation of absolute value: sqrt(x² + ε²)"""
    return ca.sqrt(x**2 + epsilon**2)

def smooth_sign_casadi(x, epsilon=1e-3):
    """Smooth approximation of sign function: tanh(x/ε)"""
    return ca.tanh(x / epsilon)

print("Creating CasADi dynamics with smooth approximations...")

In [ ]:
# HH Sport Cub S2 Parameter Values (from cyecca/models/fixedwing_4ch.py)
params = {
    # Basic physical parameters
    'thr_max': 0.56,        # Maximum thrust (N)
    'm': 0.057,             # Mass (kg)
    'S': 0.05553,           # Wing surface area (m^2)
    'rho': 1.225,           # Air density at sea level (kg/m^3)
    'g': 9.81,              # Gravitational acceleration (m/s^2)
    
    # Moments of Inertia
    'Jx': 2.0e-4,           # Roll moment of inertia (kg·m²)
    'Jy': 2.6e-4,           # Pitch moment of inertia (kg·m²)
    'Jz': 3.2e-4,           # Yaw moment of inertia (kg·m²)
    'Jxz': 0.0e-4,          # Product of inertia (kg·m²)
    
    # Geometry
    'cbar': 0.09,           # Mean aerodynamic chord (m)
    'span': 0.617,          # Wingspan (m)
    'XCG': 0.25,            # Center of gravity, nondimensional
    'XAC': 0.25,            # Aerodynamic center, nondimensional
    
    # Control Effectiveness (per radian)
    'Cm0': 0.0314,          # Zero-lift pitching moment coefficient
    'Clda': 0.10,           # Aileron control effectiveness in roll
    'Cldr': 0.05,           # Rudder control effectiveness in roll
    'Cmde': 0.9,            # Elevator control effectiveness in pitch
    'Cndr': 0.12,           # Rudder control effectiveness in yaw
    'Cnda': 0.03,           # Aileron control effectiveness in yaw
    'CYda': 0.02,           # Sideforce due to aileron deflection
    'CYdr': -0.12,          # Sideforce due to rudder deflection
    
    # Longitudinal Stability
    'CL0': 0.20,            # Lift coefficient at zero AoA
    'CLa': 5.2,             # Lift slope (per rad)
    'Cma': -0.60,           # Pitching moment due to AoA (per rad)
    'Cmq': -18.0,           # Pitch damping (per rad/s)
    'CD0': 0.09,            # Parasitic drag coefficient
    'CDCLS': 0.062,         # Lift-induced drag coefficient
    
    # Lateral-Directional Stability
    'Cnb': 0.10,            # Yaw stiffness (per rad)
    'Clp': -1.30,           # Roll damping (per rad/s)
    'Cnr': -0.12,           # Yaw damping (per rad/s)
    'Cnp': -0.10,           # Yaw damping due to roll rate
    'Clr': 0.10,            # Roll damping due to yaw rate
    'CYb': -0.65,           # Sideforce due to sideslip (per rad)
    'CYr': 0.25,            # Sideforce due to yaw rate (per rad/s)
    'CYp': 0.15,            # Sideforce due to roll rate (per rad/s)
}

# Control surface limits
DEG2RAD = np.pi / 180
max_defl_ail = 30 * DEG2RAD      # Maximum aileron deflection (rad)
max_defl_elev = 24 * DEG2RAD     # Maximum elevator deflection (rad)
max_defl_rud = 20 * DEG2RAD      # Maximum rudder deflection (rad)
alpha_stall = 20 * DEG2RAD       # Stall angle of attack (rad)

print(f"HH Sport Cub S2 parameters loaded:")
print(f"  Mass: {params['m']} kg")
print(f"  Wingspan: {params['span']} m")
print(f"  Wing area: {params['S']} m²")
print(f"  Max thrust: {params['thr_max']} N")

In [ ]:
# Define symbolic variables
# State: [px, py, pz, u, v, w, qw, qx, qy, qz, p, q, r]
x = ca.MX.sym('x', 13)
# Control: [throttle, aileron, elevator, rudder] (4-channel for Cub)
u = ca.MX.sym('u', 4)

# Extract state components
px, py, pz = x[0], x[1], x[2]
u_vel, v_vel, w_vel = x[3], x[4], x[5]  # Renamed to avoid conflict with control
qw, qx, qy, qz = x[6], x[7], x[8], x[9]
p, q, r = x[10], x[11], x[12]

# Extract control components (4-channel: Throttle, Aileron, Elevator, Rudder)
throttle, aileron, elevator, rudder = u[0], u[1], u[2], u[3]

print("Symbolic variables defined")
print("State: [px, py, pz, u, v, w, qw, qx, qy, qz, p, q, r] (13D)")
print("Control: [throttle, aileron, elevator, rudder] (4D)")

In [ ]:
# Control processing with SMOOTH saturation
# Throttle saturation (minimum 1e-3) - smooth version
throttle_sat = smooth_max_casadi(throttle, 1e-3, epsilon=1e-6)

# Control surface deflections (4-channel)
ail_rad = max_defl_ail * aileron
elev_rad = max_defl_elev * elevator
rud_rad = max_defl_rud * rudder

# Velocity saturation (SMOOTH)
vel_limit = 50.0  # Higher limit for larger aircraft
u_sat = smooth_clamp_casadi(u_vel, -vel_limit, vel_limit, epsilon=1e-3)
v_sat = smooth_clamp_casadi(v_vel, -vel_limit, vel_limit, epsilon=1e-3)
w_sat = smooth_clamp_casadi(w_vel, -vel_limit, vel_limit, epsilon=1e-3)

# Airspeed calculation with tolerance (SMOOTH)
tol_v = 0.1
V = ca.sqrt(u_sat**2 + v_sat**2 + w_sat**2)
V_safe = smooth_max_casadi(V, tol_v, epsilon=1e-3)
u_safe = smooth_max_casadi(smooth_abs_casadi(u_sat), tol_v) * smooth_sign_casadi(u_sat)

# Angle of attack and sideslip
alpha = ca.atan2(-w_sat, u_safe)
beta = ca.asin(v_sat / V_safe)

# Angle saturation (SMOOTH)
alpha_max = 45 * DEG2RAD
alpha_min = -30 * DEG2RAD
beta_max = 30 * DEG2RAD

alpha = smooth_clamp_casadi(alpha, alpha_min, alpha_max, epsilon=1e-3)
beta = smooth_clamp_casadi(beta, -beta_max, beta_max, epsilon=1e-3)

# Dynamic pressure
qbar = 0.5 * params['rho'] * V_safe**2

print("Control processing and airspeed calculation done (SMOOTH)")

In [ ]:
# Aerodynamic coefficients with stall model
# Lift coefficient with smooth stall transition
CL_linear = params['CL0'] + params['CLa'] * alpha

# Smooth stall model using tanh for gradual transition
stall_factor = (1.0 - ca.tanh((smooth_abs_casadi(alpha) - alpha_stall) / 0.1)) / 2.0
CL = CL_linear * stall_factor + params['CL0'] * (1.0 - stall_factor)

# Drag polar
CD = params['CD0'] + params['CDCLS'] * CL**2

# Sideforce coefficient (includes aileron and rudder effects)
CY = -(params['CYb'] * beta) + \
     (params['CYda'] * ail_rad / max_defl_ail) + \
     (params['CYdr'] * rud_rad / max_defl_rud) + \
     ((params['span'] / (2 * V_safe)) * ((params['CYp'] * p) + (params['CYr'] * r)))

# Aerodynamic forces in wind frame
Dw = qbar * params['S'] * CD  # Drag
Lw = qbar * params['S'] * CL  # Lift
Yw = qbar * params['S'] * CY  # Side force

print("Aerodynamic coefficients calculated (with smooth stall model)")

In [ ]:
# Rotation matrices using CasADi
def rotation_matrix_casadi(qw, qx, qy, qz):
    """Rotation matrix from quaternion"""
    R11 = 1 - 2*(qy**2 + qz**2)
    R12 = 2*(qx*qy - qw*qz)
    R13 = 2*(qx*qz + qw*qy)
    R21 = 2*(qx*qy + qw*qz)
    R22 = 1 - 2*(qx**2 + qz**2)
    R23 = 2*(qy*qz - qw*qx)
    R31 = 2*(qx*qz - qw*qy)
    R32 = 2*(qy*qz + qw*qx)
    R33 = 1 - 2*(qx**2 + qy**2)
    return ca.vertcat(
        ca.horzcat(R11, R12, R13),
        ca.horzcat(R21, R22, R23),
        ca.horzcat(R31, R32, R33)
    )

# Wind to body rotation
cos_half_alpha = ca.cos(alpha/2)
sin_half_alpha = ca.sin(alpha/2)
cos_half_beta = ca.cos(beta/2)
sin_half_beta = ca.sin(beta/2)

qw_bn = cos_half_beta * cos_half_alpha
qx_bn = sin_half_beta * sin_half_alpha
qy_bn = -cos_half_beta * sin_half_alpha
qz_bn = sin_half_beta * cos_half_alpha

R_bn = rotation_matrix_casadi(qw_bn, qx_bn, qy_bn, qz_bn)
R_nb = R_bn.T

# Transform forces from wind to body frame
F_wind = ca.vertcat(-Dw, Yw, Lw)
F_aero_b = R_bn @ F_wind

print("Rotation matrices and force transformations complete")

In [ ]:
# Thrust and gravity forces
T_b = ca.vertcat(params['thr_max'] * throttle_sat, 0, 0)

# Gravity in body frame (NO saturation - matching cyecca)
R_wb = rotation_matrix_casadi(qw, qx, qy, qz)
R_bw = R_wb.T
W_b = R_bw @ ca.vertcat(0, 0, -params['m'] * params['g'])

# Total force
F_total = F_aero_b + T_b + W_b

print("Forces calculated")

In [ ]:
# Moments (4-channel control includes aileron effects)
# Roll moment (includes both aileron and rudder)
Cl = (params['Clda'] * ail_rad) + (-1) * (params['Cldr'] * rud_rad)

# Pitch moment (includes center of gravity offset effect)
Cm = params['Cm0'] + \
     (params['Cma'] * alpha) + \
     (params['Cmde'] * elev_rad) + \
     ((params['XAC'] - params['XCG']) * CL)

# Yaw moment (includes both rudder and aileron coupling)
Cn = (params['Cnb'] * beta) + \
     (params['Cndr'] * rud_rad) + \
     (-1) * (params['Cnda'] * ail_rad)

# Basic aerodynamic moments
Mx_aero = qbar * params['S'] * params['span'] * Cl
My_aero = qbar * params['S'] * params['cbar'] * Cm
Mz_aero = qbar * params['S'] * params['span'] * Cn

# Damping moments
Mx_damp = (params['Clp'] * (params['span'] / (2 * V_safe)) * p) + \
          (params['Clr'] * (params['span'] / (2 * V_safe)) * r)
My_damp = (params['Cmq'] * (params['cbar'] / (2 * V_safe)) * q)
Mz_damp = (params['Cnp'] * (params['span'] / (2 * V_safe)) * p) + \
          (params['Cnr'] * (params['span'] / (2 * V_safe)) * r)

# Total moments
M_total = ca.vertcat(
    Mx_aero + Mx_damp,
    My_aero + My_damp,
    Mz_aero + Mz_damp
)

print("Moments calculated (4-channel control)")

In [ ]:
# Dynamics equations
# Position kinematics
v_b = ca.vertcat(u_sat, v_sat, w_sat)
p_dot = R_wb @ v_b

# Translational dynamics
omega_b = ca.vertcat(p, q, r)
omega_skew = ca.vertcat(
    ca.horzcat(0, -r, q),
    ca.horzcat(r, 0, -p),
    ca.horzcat(-q, p, 0)
)
v_b_dot = (1/params['m']) * F_total - omega_skew @ v_b

# Quaternion kinematics (with normalization)
q_norm = ca.sqrt(qw**2 + qx**2 + qy**2 + qz**2)
qw_n, qx_n, qy_n, qz_n = qw/q_norm, qx/q_norm, qy/q_norm, qz/q_norm

Omega = ca.vertcat(
    ca.horzcat(0, -p, -q, -r),
    ca.horzcat(p, 0, r, -q),
    ca.horzcat(q, -r, 0, p),
    ca.horzcat(r, q, -p, 0)
)
q_vec_dot = 0.5 * Omega @ ca.vertcat(qw_n, qx_n, qy_n, qz_n)

# Rotational dynamics (includes product of inertia)
J = ca.vertcat(
    ca.horzcat(params['Jx'], 0, params['Jxz']),
    ca.horzcat(0, params['Jy'], 0),
    ca.horzcat(params['Jxz'], 0, params['Jz'])
)
J_omega = J @ omega_b
omega_b_dot = ca.solve(J, M_total - omega_skew @ J_omega)

# Complete dynamics vector
f_casadi = ca.vertcat(
    p_dot,          # px_dot, py_dot, pz_dot
    v_b_dot,        # u_dot, v_dot, w_dot
    q_vec_dot,      # qw_dot, qx_dot, qy_dot, qz_dot
    omega_b_dot     # p_dot, q_dot, r_dot
)

print("Dynamics equations assembled")

In [ ]:
# Create CasADi functions (FAST!)
print("Creating CasADi functions...")

# Dynamics function
f_func = ca.Function('f', [x, u], [f_casadi])

# Jacobian function (automatic differentiation - very fast!)
F_jacobian = ca.jacobian(f_casadi, x)
F_func = ca.Function('F', [x, u], [F_jacobian])

In [ ]:
# Test the functions
print("Testing CasADi functions...")

# Test values (appropriate for Cub aircraft)
x_test = np.array([0, 0, -100, 15, 0, 1, 1, 0, 0, 0, 0, 0, 0])  # Sample state
u_test = np.array([0.5, 0.1, 0.1, 0.05])  # Sample control (4-channel)

try:
    # Test dynamics
    f_val = f_func(x_test, u_test)
    print(f"Dynamics evaluation successful: shape {f_val.shape}")
    
    # Test Jacobian
    F_val = F_func(x_test, u_test)
    print(f"Jacobian evaluation successful: shape {F_val.shape}")

    print(f"\nSample dynamics values (first 5 states):")
    print(f"  Position derivatives: {np.array(f_val[:3]).flatten()}")
    print(f"  Velocity derivatives: {np.array(f_val[3:6]).flatten()}")
    
except Exception as e:
    print(f"✗ Error: {e}")

# Define state and control dimensions for compatibility
X = ca.MX.sym('X_dummy', 13)  # For len(X) compatibility
U = ca.MX.sym('U_dummy', 4)   # For len(U) compatibility (4-channel)

print("\nHH Sport Cub S2 CasADi dynamics ready")